# 面试问题：Agentic RAG 怎样做多跳检索、查询规划、证据图和停止判断？

**一句话回答**：把复杂问题拆成待填充事实槽位，先检索第一跳实体，再用已验证实体构造下一跳查询；每轮更新带来源的 evidence graph，只有证据链覆盖回答所需槽位才生成。对重复查询、空增益、矛盾证据、hop/token/deadline 设置停止，输出逐 claim 引用完整路径。

本 Notebook 用小型知识库手写倒排检索、frontier、多跳状态机、证据路径、冲突版本和分层评测。

In [ ]:
from dataclasses import dataclass,field
from collections import defaultdict,deque
import hashlib,json,math,re

SEED122=12201
DOCS122={"d1":{"tenant":"T1","text":"Alice 就职于 Acme","time":10},"d2":{"tenant":"T1","text":"Acme 总部位于 上海","time":20},"d3":{"tenant":"T1","text":"上海 位于 中国","time":30},"d4":{"tenant":"T2","text":"Acme 总部位于 巴黎","time":40}}
assert len(DOCS122)==4
assert DOCS122["d1"]["tenant"]=="T1"
assert SEED122==12201

## 1. 问题先转为答案槽位与证据要求

“Alice 工作公司的总部在哪个国家”需要 `person→company→city→country` 三条关系。Planner 只声明待求槽位和允许关系，不提前编答案；高风险任务还定义最小来源数、权威与时间要求。

In [ ]:
@dataclass
class QueryState122:
    question:str; tenant:str; slots:dict=field(default_factory=lambda:{"person":"Alice","company":None,"city":None,"country":None}); evidence:list=field(default_factory=list); queries:list=field(default_factory=list); hop:int=0; status:str="searching"
    def __post_init__(self):
        if not self.question or not self.tenant: raise ValueError("query_contract")
state122=QueryState122("Alice 工作公司的总部位于哪个国家？","T1")
assert state122.slots["person"]=="Alice" and state122.slots["company"] is None
assert state122.hop==0 and state122.status=="searching"
try: QueryState122("",""); raise AssertionError("bad query accepted")
except ValueError as e: assert str(e)=="query_contract"

## 2. 检索先做 ACL，再做词项匹配

教学索引按中文/英文词项构建 postings；查询只访问当前租户文档。真实系统可用 sparse+dense+entity，但 ACL 必须在候选生成阶段执行。每个 hit 保存 doc/version/score，不能只把文本拷进上下文。

In [ ]:
tok122=lambda s:re.findall(r"[A-Za-z]+|[\u4e00-\u9fff]+",s.lower()); postings122=defaultdict(set)
for did,d in DOCS122.items():
    for t in set(tok122(d["text"])): postings122[t].add(did)
def retrieve122(query,tenant,k=3):
    scores=defaultdict(int)
    for t in tok122(query):
        for did in postings122[t]:
            if DOCS122[did]["tenant"]==tenant: scores[did]+=1
    return sorted(scores,key=lambda d:(-scores[d],-DOCS122[d]["time"],d))[:k]
assert retrieve122("Alice 公司","T1")[0]=="d1"
assert "d4" not in retrieve122("Acme 总部","T1")
assert retrieve122("未知实体","T1")==[]

## 3. 下一跳 Query 只使用已验证槽位

若 company 未知，查 person 的就职关系；company 已知后查总部；city 已知后查国家。模型可生成改写，但宿主限制关系类型和实体来源，避免用未验证幻觉继续检索导致错误放大。

In [ ]:
def next_query122(s):
    if s.slots["company"] is None: return f'{s.slots["person"]} 就职于'
    if s.slots["city"] is None: return f'{s.slots["company"]} 总部位于'
    if s.slots["country"] is None: return f'{s.slots["city"]} 位于'
    return None
assert next_query122(state122)=="Alice 就职于"
state_probe122=QueryState122("q","T1"); state_probe122.slots["company"]="Acme"; assert next_query122(state_probe122)=="Acme 总部位于"
state_probe122.slots["city"]="上海"; assert next_query122(state_probe122)=="上海 位于"

## 4. 抽取器输出关系三元组与 provenance

受控例子用规则解析三种关系。真实 NER/LLM 抽取必须有 schema、置信和人工标注 oracle；抽取事实不可脱离 doc ID。若同一关系出现多个对象，保留候选冲突，不要静默选最顺眼的答案。

In [ ]:
def triples122(did):
    text=DOCS122[did]["text"]; parts=text.split()
    if len(parts)!=3: return []
    return [(parts[0],parts[1],parts[2],did,DOCS122[did]["time"])]
assert triples122("d1")[0][:3]==("Alice","就职于","Acme")
assert triples122("d2")[0][2]=="上海"
assert triples122("d3")[0][3]=="d3"

## 5. 检索循环以新证据增益驱动

每轮生成 query、检查重复、检索、抽取与当前槽位匹配的关系；无 hit、无新事实、重复 query、hop/deadline 到达则停止/拒答。下面跑完整三跳，并记录 claim path。

In [ ]:
REL_SLOT122={"就职于":"company","总部位于":"city","位于":"country"}
def run_multihop122(s,max_hops=4):
    seen=set()
    while s.hop<max_hops:
        q=next_query122(s)
        if q is None: s.status="complete"; break
        if q in seen: s.status="cycle"; break
        seen.add(q); s.queries.append(q); hits=retrieve122(q,s.tenant); added=False
        for did in hits:
            for subj,rel,obj,source,t in triples122(did):
                slot=REL_SLOT122.get(rel)
                expected=s.slots["person"] if slot=="company" else s.slots["company"] if slot=="city" else s.slots["city"] if slot=="country" else None
                if slot and subj==expected and s.slots[slot] is None: s.slots[slot]=obj; s.evidence.append({"subject":subj,"relation":rel,"object":obj,"doc":source,"time":t}); added=True; break
            if added: break
        s.hop+=1
        if not added: s.status="no_progress"; break
    if next_query122(s) is None: s.status="complete"
    elif s.hop>=max_hops and s.status=="searching": s.status="hop_budget"
    return s
completed122=run_multihop122(QueryState122("Alice 工作公司的总部位于哪个国家？","T1"))
assert completed122.status=="complete" and completed122.slots["country"]=="中国"
assert completed122.hop==3 and len(completed122.evidence)==3
assert completed122.queries==["Alice 就职于","Acme 总部位于","上海 位于"]

## 6. Evidence Graph 让答案路径可验证

将每个三元组视为带 doc 的有向边，回答必须找到从问题实体到答案实体的完整路径。引用只列最终 d3 不够，因为 company/city 两跳同样决定答案。路径中任一文档过期、无权或冲突都会使回答失效。

In [ ]:
def evidence_path122(evidence,start,target):
    graph=defaultdict(list)
    for e in evidence: graph[e["subject"]].append((e["object"],e))
    q=deque([(start,[])]); seen={start}
    while q:
        node,path=q.popleft()
        if node==target: return path
        for nxt,e in graph[node]:
            if nxt not in seen: seen.add(nxt); q.append((nxt,path+[e]))
    return None
path122=evidence_path122(completed122.evidence,"Alice","中国")
assert [e["doc"] for e in path122]==["d1","d2","d3"]
assert evidence_path122(completed122.evidence,"Alice","巴黎") is None
assert all(DOCS122[e["doc"]]["tenant"]=="T1" for e in path122)

## 7. 冲突、版本与分支搜索

同一 `(subject, relation)` 多个 object 时，先过滤权限，再按权威/有效时间；同等级冲突则保留分支并要求更多证据或人工。Beam/frontier 必须限制宽度，避免多跳候选指数爆炸。跨租户 d4 即使更新也不可参与 T1 决策。

In [ ]:
candidates_city122=[triples122("d2")[0],triples122("d4")[0]]
visible_city122=[x for x in candidates_city122 if DOCS122[x[3]]["tenant"]=="T1"]
chosen_city122=max(visible_city122,key=lambda x:x[4])
assert chosen_city122[2]=="上海"
assert len(visible_city122)==1
assert all(x[3]!="d4" for x in visible_city122)

## 8. 分层评测 query、hop、path 和答案

指标包括每跳 Recall@K、槽位抽取准确率、平均/最大 hop、重复/空增益率、证据路径完整率、最终 EM/F1、引用正确与成本。单跳题作为 baseline，多跳、冲突、无答案、权限和长路径分别做 slice。

In [ ]:
gold_docs122={"d1","d2","d3"}; retrieved_docs122={e["doc"] for e in completed122.evidence}; path_complete122=len(gold_docs122&retrieved_docs122)/len(gold_docs122)
manifest122={"schema":1,"planner":"slot-chain-v1","retriever":"acl-inverted-v1","max_hops":4,"max_frontier":5,"stop":["complete","no_progress","cycle","budget"],"citation":"full_path"}; digest122=hashlib.sha256(json.dumps(manifest122,sort_keys=True).encode()).hexdigest()
assert path_complete122==1
assert completed122.slots=={"person":"Alice","company":"Acme","city":"上海","country":"中国"}
assert len(digest122)==64 and manifest122["citation"]=="full_path"

## 面试总结

推荐回答：**答案槽位/关系合同 → ACL 检索 → 只用验证事实生成下一跳 → schema 三元组+provenance → 新证据驱动循环 → hop/frontier/cycle/no-progress 停止 → 完整 evidence path → 分跳与端到端评测**。Agentic RAG 的价值是动态决定下一次检索，不是无限搜索。

延伸阅读：[RAG](https://arxiv.org/abs/2005.11401)、[ReAct](https://arxiv.org/abs/2210.03629)、[IRCoT 多步检索](https://arxiv.org/abs/2212.10509)。